In [1]:
import os
import optuna
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from tabpfn import TabPFNRegressor
from xgboost import XGBRegressor

optuna.logging.set_verbosity(optuna.logging.WARNING)

df = pd.read_csv("./data/final_dataset.csv")
df["planningDate_dt"] = pd.to_datetime(df["planningDate_dt"])
df = df.sort_values(by="planningDate_dt").reset_index(drop=True)

y = df["totalAssignments"]
X_matrix = df.drop(columns=["totalAssignments", "planningDate_dt"], axis=1)

X_train, X_test, y_train, y_test = train_test_split(
    X_matrix, y, test_size=0.3, random_state=42, shuffle=False
)

time_split = TimeSeriesSplit(n_splits=3)

"""
Objective Functions for Optuna Params
"""
def objective_rf(trial):
    rf_params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 300, step=50),
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", 0.5])
    }

    random_forest = RandomForestRegressor(**rf_params, random_state=42, n_jobs=-1)
    mean_absolute_errors = []

    for index, val in time_split.split(X_train):
        optuna_x_train, optuna_x_val = X_train.iloc[index], X_train.iloc[val]
        optuna_y_train, optuna_y_val = y_train.iloc[index], y_train.iloc[val]

        random_forest.fit(optuna_x_train, optuna_y_train)
        predictions = random_forest.predict(optuna_x_val)
        mean_absolute_errors.append(mean_absolute_error(optuna_y_val, predictions))

    return np.mean(mean_absolute_errors)

def objective_xgb(trial):
    xgb_params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 200, step=50),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 9),
        "subsample": trial.suggest_float("subsample", 0.4, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 1.0, 15.0),
    }

    xgb = XGBRegressor(**xgb_params, random_state=42, n_jobs=-1, objective="reg:squarederror")
    mean_absolute_errors = []

    for index, val in time_split.split(X_train):
        optuna_x_train, optuna_x_val = X_train.iloc[index], X_train.iloc[val]
        optuna_y_train, optuna_y_val = y_train.iloc[index], y_train.iloc[val]

        xgb.fit(optuna_x_train, optuna_y_train)
        predictions = xgb.predict(optuna_x_val)
        mean_absolute_errors.append(mean_absolute_error(optuna_y_val, predictions))

    return np.mean(mean_absolute_errors)

"""
Functions for ML models training and predictions
"""
def random_forest_predictions():
    print("\n-------------------------")
    print("1- Random Forest Training")

    study_random_forest = optuna.create_study(direction="minimize")
    study_random_forest.optimize(objective_rf, n_trials=50)

    print("RF Tuning completed. Best params:")
    for params, value in study_random_forest.best_params.items():
        print(f"  {params}: {value}")

    rf = RandomForestRegressor(**study_random_forest.best_params, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    rf_predictions = rf.predict(X_test)


    print(f"MAE:  {mean_absolute_error(y_test, rf_predictions)}")
    print(f"RMSE: {root_mean_squared_error(y_test, rf_predictions)}")
    print(f"R2:   {r2_score(y_test, rf_predictions)}")

    return rf_predictions


def xgboost_predictions():
    print("\n-------------------------")
    print("2- eXtream Gradient Boosting Training")

    study_xgb = optuna.create_study(direction="minimize")
    study_xgb.optimize(objective_xgb, n_trials=50)

    print("XGB Tuning completed. Best params:")
    for params, value in study_xgb.best_params.items():
        print(f"  {params}: {value}")

    xgb_model = XGBRegressor(**study_xgb.best_params, random_state=42, n_jobs=-1, objective="reg:squarederror")

    xgb_model.fit(X_train, y_train)
    xgb_predictions = xgb_model.predict(X_test)

    xgb_pred_discrete = np.round(xgb_predictions).astype(int)

    print(f"MAE:  {mean_absolute_error(y_test, xgb_predictions)}")
    print(f"RMSE: {root_mean_squared_error(y_test, xgb_predictions)}")
    print(f"R2:   {r2_score(y_test, xgb_predictions)}")

    return xgb_pred_discrete

def tabpfn_predictions():
    print("\n-------------------------")
    print("3- TabPFN Training")

    os.environ["TABPFN_TOKEN"] = "tabpfn_sk_NUj6JOOu4z6GJkEQJSM-Z67PWJPjSTwXU7pFZCZgzVg"

    tabpfn = TabPFNRegressor()
    tabpfn.fit(X_train, y_train)
    tabpfn_predictions = tabpfn.predict(X_test)

    tabpfn_pred_discrete = np.round(tabpfn_predictions).astype(int)

    print(f"MAE:  {mean_absolute_error(y_test, tabpfn_predictions):.4f}")
    print(f"RMSE: {root_mean_squared_error(y_test, tabpfn_predictions):.4f}")
    print(f"R2:   {r2_score(y_test, tabpfn_predictions):.4f}")

    return tabpfn_pred_discrete


if __name__ == "__main__":
    preds_rf = random_forest_predictions()
    preds_xgb = xgboost_predictions()
    preds_tab = tabpfn_predictions()


-------------------------
1- Random Forest Training
RF Tuning completed. Best params:
  n_estimators: 200
  max_depth: 8
  min_samples_split: 7
  max_features: 0.5
MAE:  5.517256840811414
RMSE: 7.286527279196438
R2:   0.8887271628193842

-------------------------
2- eXtream Gradient Boosting Training
XGB Tuning completed. Best params:
  n_estimators: 150
  learning_rate: 0.5065274144174339
  max_depth: 9
  subsample: 0.5653847617391201
  colsample_bytree: 0.8711255050073932
  reg_lambda: 11.787368598043589
MAE:  7.9772868156433105
RMSE: 11.143045425415039
R2:   0.7397710084915161

-------------------------
3- TabPFN Training
MAE:  3.4316
RMSE: 4.4766
R2:   0.9580
